# AIVORA Studio · Google Colab / ComfyUI 算力直连笔记本

利用 Google Colab（推荐 **L4 24GB** 或 **A100 40GB/80GB**）一键启动 **ComfyUI + ComfyUI-VideoHelperSuite + AIVORA 安全鉴权桥接 + Cloudflare Quick Tunnel**。

本笔记本会按你选择的 `MODEL_PRESET` 从 HuggingFace 自动下载真实开源权重到 `MODELS_CACHE_DIR`（开启 Google Drive 缓存后重开免重复下载），软链到 `/content/ComfyUI/models/`，并加载仓库内置的可执行 ComfyUI API Workflow JSON。第 3 步会通过 `/aivora/health` 实地校验「权重文件 + 工作流 JSON + ComfyUI 节点」三项均就绪后，再输出真实可用模型的「一键连接码」。

### 显卡档位与空间建议
- **T4 (16GB VRAM)**：推荐 `flux-schnell`（FP8 生图，约 17.2 GB 权重）
- **L4 (24GB VRAM) / A100 (40GB / 80GB VRAM)**：推荐 `flux-schnell + wan2.2-i2v-5b`（FP8 生图 + Wan 2.2 TI2V-5B 图生视频，总计约 34 GB 权重）
- **Google Drive 空间提醒**：免费版 Google Drive 仅 15 GB，放不下 17.2 GB 的 Flux 权重；如无 100GB+ 付费云盘空间，请在第 1 步取消勾选 `USE_GOOGLE_DRIVE_CACHE`（改用 Colab 实例本地磁盘）。
- **自定义更多模型**：可将任意 ComfyUI API Workflow JSON 放入 `/content/workflows/<model_id>.json`，权重放到 `/content/ComfyUI/models/` 对应目录，`/aivora/health` 校验通过后会自动纳入可用模型列表。

In [ ]:
# @title 1. 检查 GPU 显存并可选挂载 Google Drive（缓存模型权重，重开免重复下载）
USE_GOOGLE_DRIVE_CACHE = True  # @param {type:"boolean"}
MODEL_PRESET = "flux-schnell + wan2.2-i2v-5b"  # @param ["flux-schnell", "flux-schnell + wan2.2-i2v-5b"]
HF_TOKEN = ""  # @param {type:"string"}

import os
import subprocess
from pathlib import Path

try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode().strip())
except Exception as exc:
    print("未检测到 NVIDIA GPU（请点击菜单：代码执行程序 -> 更改运行时类型 -> L4 / T4 / A100）：", exc)

MODELS_CACHE_DIR = "/content/aivora_models"
if USE_GOOGLE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MODELS_CACHE_DIR = "/content/drive/MyDrive/AIVORA_Models"
    except Exception as exc:
        print("非 Colab 环境或跳过 Drive 挂载，回退至本机临时目录：", exc)

for sub in ("checkpoints", "diffusion_models", "text_encoders", "vae"):
    (Path(MODELS_CACHE_DIR) / sub).mkdir(parents=True, exist_ok=True)

SELECTED_MODELS = [m.strip() for m in MODEL_PRESET.split("+") if m.strip()]
print("已选模型预设：", SELECTED_MODELS)
print("模型缓存目录：", MODELS_CACHE_DIR)

In [ ]:
# @title 2. 安装 ComfyUI、VideoHelperSuite、工作流模板，并按预设下载与软链真实权重
import os
import shutil
import subprocess
from pathlib import Path

REPO_RAW_BASE = "https://raw.githubusercontent.com/chen11-A/aivora-studio/main/colab"
COMFY_DIR = Path("/content/ComfyUI")
COMFY_MODELS_DIR = COMFY_DIR / "models"
WORKFLOW_DIR = Path("/content/workflows")
CACHE_ROOT = Path(MODELS_CACHE_DIR)

# 1) 安装 ComfyUI 主程序与基础依赖
if not COMFY_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DIR)])

subprocess.check_call([
    "pip", "install", "-q",
    "-r", str(COMFY_DIR / "requirements.txt"),
    "fastapi", "uvicorn", "pydantic", "huggingface_hub", "imageio-ffmpeg",
])

# 2) 安装视频封装必备自定义节点 ComfyUI-VideoHelperSuite（提供 VHS_VideoCombine H.264 MP4 输出）
vhs_dir = COMFY_DIR / "custom_nodes" / "ComfyUI-VideoHelperSuite"
if not vhs_dir.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
        str(vhs_dir),
    ])
vhs_req = vhs_dir / "requirements.txt"
if vhs_req.is_file():
    subprocess.check_call(["pip", "install", "-q", "-r", str(vhs_req)])

# 3) 下载 cloudflared 临时隧道客户端
cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    subprocess.check_call([
        "curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", str(cf_bin),
    ])
    cf_bin.chmod(0o755)

# 4) 拉取本仓库的 aivora_worker.py 及对应模型的标准 ComfyUI API Workflow JSON
subprocess.check_call([
    "curl", "-fsSL", f"{REPO_RAW_BASE}/aivora_worker.py", "-o", "/content/aivora_worker.py",
])
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)
for model_id in SELECTED_MODELS:
    wf_target = WORKFLOW_DIR / f"{model_id}.json"
    subprocess.check_call([
        "curl", "-fsSL", f"{REPO_RAW_BASE}/workflows/{model_id}.json", "-o", str(wf_target),
    ])
    print(f"✓ 已就绪工作流模板: {wf_target}")

# 5) 按 MODEL_PRESET 从 HuggingFace 下载真实权重到 MODELS_CACHE_DIR，并软链到 ComfyUI/models/
#    格式: (hf_repo_id, hf_filename, comfy_subdir, local_filename)
WEIGHT_MANIFEST = {
    "flux-schnell": [
        (
            "Comfy-Org/flux1-schnell",
            "flux1-schnell-fp8.safetensors",
            "checkpoints",
            "flux1-schnell-fp8.safetensors",
        ),
    ],
    "wan2.2-i2v-5b": [
        (
            "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
            "split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",
            "diffusion_models",
            "wan2.2_ti2v_5B_fp16.safetensors",
        ),
        (
            "Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
            "split_files/vae/wan2.2_vae.safetensors",
            "vae",
            "wan2.2_vae.safetensors",
        ),
        (
            "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
            "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
            "text_encoders",
            "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        ),
    ],
}

from huggingface_hub import hf_hub_download

HF_TMP_CACHE = Path("/content/.hf_tmp_cache")
hf_token = HF_TOKEN.strip() or None

def ensure_weight(repo_id: str, hf_filename: str, subdir: str, local_name: str) -> None:
    cache_file = CACHE_ROOT / subdir / local_name
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    comfy_link = COMFY_MODELS_DIR / subdir / local_name
    comfy_link.parent.mkdir(parents=True, exist_ok=True)

    if cache_file.is_file() and cache_file.stat().st_size > 50_000_000:
        size_gb = cache_file.stat().st_size / (1024 ** 3)
        print(f"✓ 复用已缓存权重（免重复下载）: {subdir}/{local_name} ({size_gb:.2f} GB)")
    else:
        print(f"↓ 正在从 HuggingFace 下载: {repo_id}/{hf_filename} ...")
        HF_TMP_CACHE.mkdir(parents=True, exist_ok=True)
        downloaded = Path(
            hf_hub_download(
                repo_id=repo_id,
                filename=hf_filename,
                cache_dir=str(HF_TMP_CACHE),
                token=hf_token,
            )
        ).resolve()
        tmp_dest = cache_file.with_suffix(cache_file.suffix + ".part")
        try:
            if downloaded.stat().st_dev == cache_file.parent.stat().st_dev:
                shutil.move(str(downloaded), str(tmp_dest))
            else:
                shutil.copy2(str(downloaded), str(tmp_dest))
            os.replace(tmp_dest, cache_file)
        finally:
            if tmp_dest.exists():
                tmp_dest.unlink(missing_ok=True)
            shutil.rmtree(HF_TMP_CACHE, ignore_errors=True)
        size_gb = cache_file.stat().st_size / (1024 ** 3)
        print(f"✓ 下载完成并缓存: {cache_file} ({size_gb:.2f} GB)")

    # 幂等创建软链：/content/ComfyUI/models/<subdir>/<local_name> -> MODELS_CACHE_DIR/<subdir>/<local_name>
    if comfy_link.is_symlink() or comfy_link.exists():
        comfy_link.unlink()
    comfy_link.symlink_to(cache_file)
    print(f"  ↳ 已软链至 ComfyUI: {comfy_link} -> {cache_file}")

for model_id in SELECTED_MODELS:
    for repo_id, hf_filename, subdir, local_name in WEIGHT_MANIFEST.get(model_id, []):
        ensure_weight(repo_id, hf_filename, subdir, local_name)

print("\n全部依赖、工作流 JSON 与权重软链准备完毕，请继续运行第 3 步！")

In [ ]:
# @title 3. 一键启动 ComfyUI + AIVORA 鉴权 Worker + Cloudflare 临时隧道
import base64
import json
import os
import re
import secrets
import subprocess
import time
import urllib.request
from pathlib import Path

AIVORA_TOKEN = secrets.token_urlsafe(24)
procs = []

def stop_all() -> None:
    for proc in reversed(procs):
        if proc.poll() is None:
            proc.terminate()
    for proc in reversed(procs):
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()

def tail_log(path: str, max_chars: int = 800) -> str:
    p = Path(path)
    if not p.is_file():
        return "(无日志)"
    return p.read_text(encoding="utf-8", errors="ignore")[-max_chars:].strip()

try:
    # 1) 启动 ComfyUI 后端并等待节点注册完成（崩溃快速失败）
    comfy_log = open("/content/comfy.log", "w")
    comfy_proc = subprocess.Popen(
        ["python3", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
        stdout=comfy_log,
        stderr=subprocess.STDOUT,
    )
    procs.append(comfy_proc)

    comfy_ready = False
    for _ in range(90):
        if comfy_proc.poll() is not None:
            raise RuntimeError(f"ComfyUI 进程异常退出（code={comfy_proc.returncode}）：\n{tail_log('/content/comfy.log')}")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=3) as resp:
                if resp.status == 200:
                    comfy_ready = True
                    break
        except Exception:
            time.sleep(2)

    if not comfy_ready:
        raise RuntimeError(f"ComfyUI 启动超时：\n{tail_log('/content/comfy.log')}")

    # 2) 启动 AIVORA 安全鉴权桥接（不硬编码假模型列表，由 /aivora/health 实地校验权重+工作流+节点）
    env = os.environ.copy()
    env["AIVORA_WORKER_TOKEN"] = AIVORA_TOKEN
    env["AIVORA_WORKFLOW_DIR"] = "/content/workflows"
    env["COMFY_MODELS_DIR"] = "/content/ComfyUI/models"
    env.pop("AIVORA_MODELS", None)
    worker_log = open("/content/worker.log", "w")
    worker_proc = subprocess.Popen(
        ["uvicorn", "aivora_worker:app", "--app-dir", "/content", "--host", "127.0.0.1", "--port", "8199"],
        env=env,
        stdout=worker_log,
        stderr=subprocess.STDOUT,
    )
    procs.append(worker_proc)

    # 3) 实地调用 /aivora/health 获取真正就绪的模型列表
    health_data = None
    for _ in range(30):
        if worker_proc.poll() is not None:
            raise RuntimeError(f"AIVORA Worker 进程异常退出（code={worker_proc.returncode}）：\n{tail_log('/content/worker.log')}")
        try:
            req = urllib.request.Request(
                "http://127.0.0.1:8199/aivora/health",
                headers={"Authorization": f"Bearer {AIVORA_TOKEN}"},
            )
            with urllib.request.urlopen(req, timeout=5) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                if data.get("ok") and data.get("comfy_reachable"):
                    health_data = data
                    break
        except Exception:
            time.sleep(1)

    if not health_data:
        raise RuntimeError(f"AIVORA Worker 健康检查未通过：\n{tail_log('/content/worker.log')}")

    ready_models = health_data.get("models") or []
    not_ready = health_data.get("not_ready") or {}

    if not_ready:
        print("未就绪模型诊断（不会写入连接码）：")
        for mid, reasons in not_ready.items():
            print(f"  - {mid}: {'；'.join(reasons)}")

    if not ready_models:
        raise RuntimeError("没有任何模型通过权重+工作流+节点就绪校验，请先重新运行第 2 步并检查上方诊断。")

    # 4) 启动 Cloudflare Quick Tunnel（输出落盘至 /content/tunnel.log，彻底避免 PIPE 64KB 缓冲区死锁）
    tunnel_log_path = "/content/tunnel.log"
    tunnel_log = open(tunnel_log_path, "w")
    tunnel_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8199", "--no-autoupdate"],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
    )
    procs.append(tunnel_proc)

    public_url = None
    deadline = time.time() + 45
    while time.time() < deadline:
        if tunnel_proc.poll() is not None:
            raise RuntimeError(f"cloudflared 异常退出（code={tunnel_proc.returncode}）：\n{tail_log(tunnel_log_path)}")
        text = Path(tunnel_log_path).read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
        if match:
            public_url = match.group(0)
            break
        time.sleep(0.5)

    if not public_url:
        raise RuntimeError(f"45 秒内未能获取 Cloudflare 临时公网地址：\n{tail_log(tunnel_log_path)}")

    connect_payload = {
        "url": public_url,
        "token": AIVORA_TOKEN,
        "models": ready_models,
        "name": f"Google Colab ({health_data.get('gpu', 'GPU')})",
    }
    connect_code = "AIVORA-COMFY::" + base64.b64encode(
        json.dumps(connect_payload, ensure_ascii=False).encode("utf-8")
    ).decode("ascii")

    print("=" * 72)
    print("  AIVORA Studio 算力 Worker 已就绪！")
    print(f"  GPU 型号     : {health_data.get('gpu', 'GPU Worker')}")
    print(f"  已验证模型   : {', '.join(ready_models)}")
    print("  【最简单用法】整段复制下方「一键连接码」，粘贴进模型中心即可自动填充：")
    print("-" * 72)
    print(connect_code)
    print("-" * 72)
    print(f"  Base URL     : {public_url}")
    print(f"  Access Token : {AIVORA_TOKEN}")
    print("=" * 72)
    print("提示：保持本单元格运行即可；Colab 重开后点模型中心「编辑」再粘一次新连接码即可。")

    # 5) 持续守护三个核心进程，任一异常退出立即告警
    while True:
        if comfy_proc.poll() is not None:
            raise RuntimeError(f"ComfyUI 意外退出（可能显存不足 OOM，code={comfy_proc.returncode}）：\n{tail_log('/content/comfy.log')}")
        if worker_proc.poll() is not None:
            raise RuntimeError(f"AIVORA Worker 意外退出（code={worker_proc.returncode}）：\n{tail_log('/content/worker.log')}")
        if tunnel_proc.poll() is not None:
            raise RuntimeError(f"Cloudflare 隧道意外断开（code={tunnel_proc.returncode}）：\n{tail_log(tunnel_log_path)}")
        time.sleep(2)
except KeyboardInterrupt:
    print("收到手动停止信号，正在清理后台进程…")
finally:
    stop_all()